> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 4.2 - APIs


## Exercícios 🔭🌌🪐

Vamos acessar as APIs da NASA para ver algumas imagens interessantes capturadas universo afora!

#### Q1.
Crie uma chave no site oficial:
* https://api.nasa.gov

Vamos armazenar a chave de forma segura! <br>
Salve a sua chave em um arquivo `key.json` na forma:
`API_KEY=SUA_CHAVE`

Adicione o nome do arquivo `key.json` ao `.gitignore` do repositório que você fará upload da atividade.
Para isso basta abrir o arquivo `.gitignore` na pasta raíz do repositório (ou criar um caso ele não exista). Dentro do arquivo, apenas adiciona o nome do arquivo que deseja ignorar.

#### Q2. 🛰 Astronomy Picture of the Day (APOD) 🌌
> Antes de fazer os exercícios, devo te lembrar que existem limites de acesso às APIs, descritas na página principal, portanto pega leve na tentativa e erro na hora de testar seu código.

<img width=500 src=https://apod.nasa.gov/apod/image/2407/M24-HaLRGB-RC51_1024.jpg>

A primeira API que acessaremos é a mais popular de todas: astronomy picture of the day (foto astronômica do dia).

Faça uma requisição GET para a URL da API que retorna a imagem do dia! Essa é fácil já que são os valores padrão da rota principal:
* URL base: `'https://api.nasa.gov/planetary/apod'`
* Endpoint: não precisa preencher, acessaremos a raíz da API.
* Query params: preencha `api_key` com a sua chave de autenticação. Se animar mexer em outros parâmetros veja [a documentação](https://api.nasa.gov).

Ao receber a resposta (um json), você deve:
* Imprimir os campos `copyright` e `explanation`
* Com as biblioteca scikit-images e matplotlib, apresente a imagem a partir do campo `url` ou `hdurl`, e preencha o título do plot com o campo `title` do json. Uma dica de código a seguir.
```python
from skimage import io
img = io.imread(url)
## plot a matriz img com matplotlib (imshow)
```   

In [8]:
import requests
import matplotlib.pyplot as plt
from skimage import io
import os

# Ajuste a pasta de trabalho, se necessário
try:
    os.chdir(r"D:\Google Drive\Meu Drive\Onedrive\Desenvolve BD\Python II\desenvolve-python-II\Módulo III\Projeto NASA")
except:
    pass

# 1. Carregar chave de forma segura
with open('key.json', 'r') as f:
    api_key = f.readline().strip().replace('API_KEY=', '')

# 2. Requisição
url_base = 'https://api.nasa.gov/planetary/apod'
params = {'api_key': api_key}

response = requests.get(url_base, params=params)

# 3. Processamento com verificação de erro
if response.status_code == 200:
    data = response.json()
    
    print(f"Copyright: {data.get('copyright', 'Não disponível')}")
    print(f"\nExplicação:\n{data.get('explanation', 'Explicação não disponível')}")

    url_imagem = data.get('hdurl', data.get('url'))
    
    if url_imagem:
        img = io.imread(url_imagem)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.title(data.get('title', 'NASA APOD'))
        plt.axis('off')
        plt.show()
    else:
        print("URL da imagem não encontrada no JSON.")
else:
    print(f"Erro na conexão com a API. Status Code: {response.status_code}")
    print("Aguarde alguns minutos e tente novamente, pois o servidor pode estar instável.")

Erro na conexão com a API. Status Code: 500
Aguarde alguns minutos e tente novamente, pois o servidor pode estar instável.


#### Q3. Limites
A partir da resposta da query anterios, imprima o header da resposta e consulte os atributos:
* X-RateLimit-Limit: o limite total de requisições da sua chave de API
* X-RateLimit-Remaining: o limite restante de requisições da sua chave de API

In [9]:
# Acessando os headers da última requisição feita
# O método .get() é usado aqui para evitar erro caso o header não exista
limite_total = response.headers.get('X-RateLimit-Limit')
limite_restante = response.headers.get('X-RateLimit-Remaining')

print(f"Limite total de requisições: {limite_total}")
print(f"Limite restante de requisições: {limite_restante}")

Limite total de requisições: 4000
Limite restante de requisições: 3996


### Q4. Mars Rover Photos 🚀🚙 📷

<img width=500 src=https://www.nasa.gov/wp-content/uploads/2019/10/pia23378-16.jpg>

Essa API retorna dados (incluindo imagens capturadas) sobre os veículos que hoje habitam o planeta Marte. São os rovers `opportunity`, `spirit` e o mais famoso, o `curiosity` (da foto acima).

Antes de requisitar imagens, vamos ver o relatório de dados coletados por um deles, o `curiosity`. Isso vai nos ajudar a montar a query de imagens coletadas.

Faça uma requisição GET para a seguinte URL:
* URL base: `'https://api.nasa.gov/mars-photos/api/v1'`
* endpoint: `'/manifests/{nome_do_rover}'`
* query parameters: preencha `api_key` com a sua chave de autenticação.

Extraia o json da resposta retornada. O campo principal é o `'photo_manifest'`, do qual queremos acessar os seguintes valores:
* `max_sol`: Máximo "dia marciano" de coleta de fotos. O dia marciano tem 24 horas, 39 minutos e 35 segundos.
* `max_date`: Última data terrestre de coleta de fotos, na forma `'aaaa-mm-dd'`.

Imprima esses dois atributos da resposta e os use no próximo exercício para coletar as fotos mais recentes tiradas. 

In [11]:
url = 'https://api.nasa.gov/mars-photos/api/v1/manifests/curiosity'
params = {'api_key': api_key}

# Faz a requisição e checa se o servidor respondeu corretamente
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    manifest = data['photo_manifest']
    
    max_sol = manifest['max_sol']
    max_date = manifest['max_date']
    
    print(f"Max Sol: {max_sol}")
    print(f"Max Date: {max_date}")
else:
    # Caso a API esteja fora do ar ou negue o acesso
    print(f"Erro na requisição: {response.status_code}")
    print("Resposta bruta:", response.text)

Erro na requisição: 404
Resposta bruta: <!DOCTYPE html>
<html>
  <head>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta charset="utf-8">
    <title>No such app</title>
    <style media="screen">
      html,body,iframe {
        margin: 0;
        padding: 0;
      }
      html,body {
        height: 100%;
        overflow: hidden;
      }
      iframe {
        width: 100%;
        height: 100%;
        border: 0;
      }
    </style>
  </head>
  <body>
    <iframe src="//www.herokucdn.com/error-pages/no-such-app.html"></iframe>
  </body>
</html>



#### Q5.

Faça uma requisição GET para a URL da API que retorna links para as imagens coletadas pelos rovers.

* URL base: `'https://api.nasa.gov/mars-photos/api/v1'`
* Endpoint: `/rovers/{nome_do_rover}/photos`
* Query params sugeridos: 
    * `api_key`: sua chave de autenticação.
    * `sol`: dia marciano que deseja coletar (de 0 a `max_sol` coletado anteriormente)
    * `page`: você pode paginar entre as respostas! São retornados 25 resultados por página.

A resposta esperada estará no formato a seguir, uma lista no campo `'photos'` onde cada item é um dicionário com os dados da foto tirada. Dentre os dados há o campo `camera` indicando qual das câmeras do rover tirou a foto. As fotos mais interessantes (na minha opinião, claro) são das câmeras de navegação (`"name": "NAVCAM"`) e as de prevenção de colisão (frente: `"name": "FHAZ"` e trás `"name": "RHAZ"`) onde dá pra ver partes do robô!

**Seu trabalho é**:
* Paginar a requisição acima até que a resposta seja `None`
* Escolher uma ou mais câmeras (ex: `NAVCAM`, `FHAZ`, `RHAZ`), e em um laço de repetição plotar todas as imagens retornadas daquela câmera. Use novamente as bibliotecas scikit-image e matplotlib. 
  * O título da imagem deve ter a página da requisição, nome da câmera e id da imagem.

```json
{
  "photos": [
    {
      "id": 1228212,
      "sol": 4102,
      "camera": {
        "id": 20,
        "name": "FHAZ",
        "rover_id": 5,
        "full_name": "Front Hazard Avoidance Camera"
      },
      "img_src": "https://mars.nasa.gov/msl-raw-images/proj/msl/redops/ods/surface/sol/04102/opgs/edr/fcam/FLB_761645828EDR_F1060660FHAZ00302M_.JPG",
      "earth_date": "2024-02-19",
      "rover": {
        "id": 5,
        "name": "Curiosity",
        ...
      }
    }
    {
      "id": 1228213,
      "sol": 4102, 
      ...
    }
```



In [13]:
import requests
import matplotlib.pyplot as plt
from skimage import io

nome_rover = 'curiosity'
camera_desejada = 'NAVCAM'
sol = 1000 
page = 1

while True:
    url = f'https://api.nasa.gov/mars-photos/api/v1/rovers/{nome_rover}/photos'
    params = {'api_key': api_key, 'sol': sol, 'page': page, 'camera': camera_desejada}
    
    response = requests.get(url, params=params)
    
    # Verifica se a conexão foi bem-sucedida (Status 200)
    if response.status_code == 200:
        data = response.json()
        
        if not data.get('photos'):
            break
            
        for item in data['photos']:
            img_url = item['img_src']
            img = io.imread(img_url)
            
            plt.figure(figsize=(8, 8))
            plt.imshow(img)
            plt.title(f"Pág: {page} | Câm: {item['camera']['name']} | ID: {item['id']}")
            plt.axis('off')
            plt.show()
        
        page += 1
    else:
        print(f"Erro na página {page}. Status: {response.status_code}")
        break # Para o loop se a API falhar

Erro na página 1. Status: 404
